# imports

In [1]:
# ===========import============
from pathlib import Path
import time
from data_provider import (
    set_seed,
    SEED,
    compute_hashes,
    get_balanced_sampler,
    data_cleaner,
    get_transform,
    make_clean_dataset,
    make_loader,
    to_str_list,
    ROOT,
    unclean_path,
    train_path,
    test_path,
)

from run_experiment import run_experiment
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights

# ===========control-randomness=============
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# =========load-data============
if unclean_path.exists():
    print("unclean path exists")
else:
    raise FileNotFoundError("unclean path does not exist")

if train_path.exists():
    print("train path exists")
else:
    raise FileNotFoundError("train path does not exist")

if test_path.exists():
    print("test path exists")
else:
    raise FileNotFoundError("test path does not exist")


cleaner_output = data_cleaner(train_path, test_path, unclean_path)

dataset_train, dataset_train_base, dataset_test = make_clean_dataset(
    unclean_path, train_path, test_path, cleaner_output
)

_, train_base_loader, validation_loader, test_loader = make_loader(
    dataset_train, dataset_train_base, dataset_test
)

unclean path exists
train path exists
test path exists


# BackBone Freeze

In [3]:
# ============paths===========
csv_report_path = ROOT / "reports" / "csv" / "resnet_freeze"
plot_report_path = ROOT / "reports" / "plots" / "resnet_freeze"
models_path = ROOT / "models" / "resnet_freeze"

csv_report_path.mkdir(parents=True, exist_ok=True)
plot_report_path.mkdir(parents=True, exist_ok=True)
models_path.mkdir(parents=True, exist_ok=True)

# ==========train-requires-obj===========
set_seed(SEED)
classes = dataset_train.classes

resnet = resnet18(weights=ResNet18_Weights.DEFAULT).to(DEVICE)

# freeze whole model
for param in resnet.parameters():
    param.requires_grad = False

# new classifier
in_features = resnet.fc.in_features

resnet.fc = nn.Linear(in_features, out_features=len(classes)).to(DEVICE)
resnet.trainable_modules = [resnet.fc]
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-3
)

start_time = time.perf_counter()
history, best_epoch, model = run_experiment(
    model=resnet,
    train_loader=train_base_loader,
    val_loader=validation_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=DEVICE,
    epochs=8,
    class_names=classes,
)

elapsed_time = time.perf_counter() - start_time

minutes = int(elapsed_time // 60)
seconds = elapsed_time % 60

print(f"Training completed in: {minutes}m {seconds:.2f}s ({elapsed_time:.2f}s)")

# ============save-reports-->csv==============
csv_report = pd.DataFrame(
    {
        "train_f1_score": history["train_f1"],
        "train_accuracy": history["train_accuracy"],
        "val_f1_score": history["val_f1"],
        "val_accuracy": history["val_accuracy"],
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"],
    }
)

cf_report_val = pd.DataFrame(history["val_cf_report"][best_epoch]).transpose().round(4)
cf_report_val.to_csv(csv_report_path / "resnet_freeze_CF_VAL.csv", index_label="class")
cf_report_train = (
    pd.DataFrame(history["train_cf_report"][best_epoch]).transpose().round(4)
)

cf_report_train.to_csv(
    csv_report_path / "resnet_freeze_CF_train.csv", index_label="class"
)

csv_report.to_csv(csv_report_path / "resnet_freeze.csv", index=False)


dsp = ConfusionMatrixDisplay(
    confusion_matrix=history["val_confusion_matrix"][best_epoch],
    display_labels=dataset_train.classes,
)
dsp.plot(xticks_rotation=45)
dsp.ax_.set_title(f"Best Epoch: {best_epoch+1}")
plt.grid(False)
plt.savefig(plot_report_path / "resnet_freeze_val.png")
plt.close()

dsp = ConfusionMatrixDisplay(
    confusion_matrix=history["train_confusion_matrix"][best_epoch],
    display_labels=dataset_train.classes,
)
dsp.plot(xticks_rotation=45)

dsp.ax_.set_title(f"Best Epoch: {best_epoch+1}")
plt.grid(False)
plt.savefig(plot_report_path / "resnet_freeze_train.png")
plt.close()


torch.save(model.state_dict(), models_path / "resnet_freeze.pth")
print("Saved Successfully")

Training completed in: 14m 9.44s (849.44s)
Saved Successfully


# Resnet fine tune

In [4]:
# ============paths===========
csv_report_path = ROOT / "reports" / "csv" / "resnet_finetune"
plot_report_path = ROOT / "reports" / "plots" / "resnet_finetune"
models_path = ROOT / "models" / "resnet_finetune"

csv_report_path.mkdir(parents=True, exist_ok=True)
plot_report_path.mkdir(parents=True, exist_ok=True)
models_path.mkdir(parents=True, exist_ok=True)

# ==========train-requires-obj===========
set_seed(SEED)
classes = dataset_train.classes

resnet = resnet18(weights=ResNet18_Weights.DEFAULT).to(DEVICE)

# freeze whole model
for param in resnet.parameters():
    param.requires_grad = False

for param in resnet.layer4.parameters():
    param.requires_grad = True


# new classifier
in_features = resnet.fc.in_features

resnet.fc = nn.Linear(in_features, out_features=len(classes)).to(DEVICE)

resnet.trainable_modules = [resnet.layer4, resnet.fc]
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {"params": resnet.layer4.parameters(), "lr": 1e-4},
        {"params": resnet.fc.parameters(), "lr": 1e-3},
    ]
)

start_time = time.perf_counter()
history, best_epoch, model = run_experiment(
    model=resnet,
    train_loader=train_base_loader,
    val_loader=validation_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=DEVICE,
    epochs=8,
    class_names=classes,
)

elapsed_time = time.perf_counter() - start_time

minutes = int(elapsed_time // 60)
seconds = elapsed_time % 60

print(f"Training completed in: {minutes}m {seconds:.2f}s ({elapsed_time:.2f}s)")

# ============save-reports-->csv==============
csv_report = pd.DataFrame(
    {
        "train_f1_score": history["train_f1"],
        "train_accuracy": history["train_accuracy"],
        "val_f1_score": history["val_f1"],
        "val_accuracy": history["val_accuracy"],
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"],
    }
)

cf_report_val = pd.DataFrame(history["val_cf_report"][best_epoch]).transpose().round(4)
cf_report_val.to_csv(
    csv_report_path / "resnet_finetune_CF_VAL.csv", index_label="class"
)
cf_report_train = (
    pd.DataFrame(history["train_cf_report"][best_epoch]).transpose().round(4)
)

cf_report_train.to_csv(
    csv_report_path / "resnet_finetune_CF_train.csv", index_label="class"
)

csv_report.to_csv(csv_report_path / "resnet_finetune.csv", index=False)


dsp = ConfusionMatrixDisplay(
    confusion_matrix=history["val_confusion_matrix"][best_epoch],
    display_labels=dataset_train.classes,
)
dsp.plot(xticks_rotation=45)
dsp.ax_.set_title(f"Best Epoch: {best_epoch+1}")
plt.grid(False)
plt.savefig(plot_report_path / "resnet_finetune_val.png")
plt.close()

dsp = ConfusionMatrixDisplay(
    confusion_matrix=history["train_confusion_matrix"][best_epoch],
    display_labels=dataset_train.classes,
)
dsp.plot(xticks_rotation=45)

dsp.ax_.set_title(f"Best Epoch: {best_epoch+1}")
plt.grid(False)
plt.savefig(plot_report_path / "resnet_finetune_train.png")
plt.close()


torch.save(model.state_dict(), models_path / "resnet_finetune.pth")
print("Saved Successfully")

Training completed in: 17m 28.25s (1048.25s)
Saved Successfully
